### Step 1: Initialize Spark and Load Data
In this first step, we set up our environment by creating a **SparkSession**. This is the entry point for any PySpark application. We then load our dataset from a CSV file.

*   `header=True`: Tells Spark the first row contains column names.
*   `inferSchema=True`: Automatically detects if a column is a number or text.

In [33]:
# Import required libraries
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ChurnML").getOrCreate()

# Load dataset
df = spark.read.csv("/content/drive/MyDrive/Big Data Foundation/churn_dataset.csv", header=True, inferSchema=True)

# Show data
df.show(5)

+-----------+---+--------------+-------------+------------+-----------------+-------------+----------------------+-----------------+------+------+-----------------+-----+
|customer_id|age|monthly_income|tenure_months|num_products|avg_monthly_spend|late_payments|customer_support_calls|internet_usage_gb|gender|region|subscription_type|churn|
+-----------+---+--------------+-------------+------------+-----------------+-------------+----------------------+-----------------+------+------+-----------------+-----+
|    1082857| 21|         74168|           14|           1|             8444|            8|                     6|              301|Female| North|         Standard|    0|
|      27692| 21|         83710|           29|           2|             9791|            9|                     4|               56|  Male| North|         Standard|    0|
|     269938| 18|         58358|           10|           2|             8217|            1|                     7|              169|Female| North

### Step 2: Exploratory Data Analysis (EDA)
Before building a model, we must 'get to know' our data. We check the **Schema** to see the data types, count the total number of records, and check the **Target Distribution** (Churn). In this case, our dataset is perfectly balanced with an equal number of churned and stayed customers.

In [34]:
# Schema (data types)
df.printSchema()

# Count rows
df.count()

# Check target distribution
df.groupBy("churn").count().show()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- monthly_income: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- num_products: integer (nullable = true)
 |-- avg_monthly_spend: integer (nullable = true)
 |-- late_payments: integer (nullable = true)
 |-- customer_support_calls: integer (nullable = true)
 |-- internet_usage_gb: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- region: string (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- churn: integer (nullable = true)

+-----+------+
|churn| count|
+-----+------+
|    1|205929|
|    0|205929|
+-----+------+



### Step 3: Data Cleaning and Null Check
Real-world data is often messy. We use `describe()` to look at statistical summaries (like mean and max) to spot outliers. We also perform a **Null Check** to ensure there are no missing values, as machine learning algorithms cannot process empty data points.

In [35]:
df.describe().show()

+-------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+----------------------+-----------------+------+------+-----------------+------------------+
|summary|      customer_id|               age|    monthly_income|     tenure_months|      num_products|avg_monthly_spend|    late_payments|customer_support_calls|internet_usage_gb|gender|region|subscription_type|             churn|
+-------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+----------------------+-----------------+------+------+-----------------+------------------+
|  count|           411858|            411858|            411858|            411858|            411858|           411858|           411858|                411858|           411858|411858|411858|           411858|            411858|
|   mean|599543.2146395117|  38.5114165561917|54982.788565476454|  25.72

In [36]:
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-----------+---+--------------+-------------+------------+-----------------+-------------+----------------------+-----------------+------+------+-----------------+-----+
|customer_id|age|monthly_income|tenure_months|num_products|avg_monthly_spend|late_payments|customer_support_calls|internet_usage_gb|gender|region|subscription_type|churn|
+-----------+---+--------------+-------------+------------+-----------------+-------------+----------------------+-----------------+------+------+-----------------+-----+
|          0|  0|             0|            0|           0|                0|            0|                     0|                0|     0|     0|                0|    0|
+-----------+---+--------------+-------------+------------+-----------------+-------------+----------------------+-----------------+------+------+-----------------+-----+



### Step 4: Categorical Encoding (StringIndexer)
Computers only understand numbers. We use `StringIndexer` to convert text columns like 'Gender' or 'Region' into numerical labels (e.g., Male = 0, Female = 1).

In [37]:
from pyspark.ml.feature import StringIndexer

# Convert categories into index numbers
indexers = [
    StringIndexer(inputCol="gender", outputCol="gender_index"),
    StringIndexer(inputCol="region", outputCol="region_index"),
    StringIndexer(inputCol="subscription_type", outputCol="subscription_index")
]

for indexer in indexers:
    df = indexer.fit(df).transform(df)

### Step 5: Feature Engineering (OneHotEncoding & VectorAssembler)
To help the model learn better, we use **OneHotEncoding** to turn our indexed categories into binary vectors. Finally, we use the `VectorAssembler` to merge all individual features into a single column called **'features'**. This is a requirement for all PySpark ML algorithms.

In [38]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=["gender_index", "region_index", "subscription_index"],
    outputCols=["gender_vec", "region_vec", "subscription_vec"]
)

df = encoder.fit(df).transform(df)

In [39]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "age", "monthly_income", "tenure_months", "num_products",
    "avg_monthly_spend", "late_payments", "customer_support_calls",
    "internet_usage_gb",
    "gender_vec", "region_vec", "subscription_vec"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df = assembler.transform(df)

### Step 6: Data Splitting
We split our data into a **Training Set (80%)** and a **Test Set (20%)**. The model learns patterns from the training set, and we use the test set to evaluate if the model can predict accurately on data it has never seen before.

In [40]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 329358
Test: 82500


### Step 7: Model Training
We train four different types of classifiers to see which one handles this data best:
1.  **Logistic Regression**: A linear model used as a baseline.
2.  **Decision Tree**: A model that makes decisions based on simple rules.
3.  **Random Forest**: An ensemble method that combines many decision trees for better accuracy.
4.  **Linear SVC (SVM)**: A model that finds the best boundary between classes.

In [41]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="churn")

lr_model = lr.fit(train_df)

lr_pred = lr_model.transform(test_df)

In [42]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(featuresCol="features", labelCol="churn")

dt_model = dt.fit(train_df)

dt_pred = dt_model.transform(test_df)

In [43]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="churn", numTrees=50)

rf_model = rf.fit(train_df)

rf_pred = rf_model.transform(test_df)

In [44]:
from pyspark.ml.classification import LinearSVC

svm = LinearSVC(featuresCol="features", labelCol="churn")

svm_model = svm.fit(train_df)

svm_pred = svm_model.transform(test_df)

### Step 8: Evaluation on Test Data (AUC)
We use the **Area Under the Curve (AUC)** to measure performance. An AUC of 1.0 is a perfect model, while 0.5 is no better than a random guess. This tells us how well our models distinguish between churn and non-churn.

In [45]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="churn")

# Evaluate all models
lr_auc = evaluator.evaluate(lr_pred)
dt_auc = evaluator.evaluate(dt_pred)
rf_auc = evaluator.evaluate(rf_pred)
svm_auc = evaluator.evaluate(svm_pred)

print("Logistic Regression AUC:", lr_auc)
print("Decision Tree AUC:", dt_auc)
print("Random Forest AUC:", rf_auc)
print("SVM AUC:", svm_auc)

Logistic Regression AUC: 0.8529618890230132
Decision Tree AUC: 0.9934791820944405
Random Forest AUC: 0.9978693621799106
SVM AUC: 0.8531343665660119


### Step 9: Evaluation on Test Data (Accuracy)
We also calculate **Accuracy**, which is the simple percentage of correct predictions. This is often the easiest metric to explain to stakeholders.

In [46]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_eval = MulticlassClassificationEvaluator(labelCol="churn", metricName="accuracy")

print("LR Accuracy:", acc_eval.evaluate(lr_pred))
print("DT Accuracy:", acc_eval.evaluate(dt_pred))
print("RF Accuracy:", acc_eval.evaluate(rf_pred))
print("SVM Accuracy:", acc_eval.evaluate(svm_pred))

LR Accuracy: 0.766630303030303
DT Accuracy: 0.9900242424242425
RF Accuracy: 0.9962545454545455
SVM Accuracy: 0.7664484848484848


In [47]:
results = [
    ("Logistic Regression", lr_auc),
    ("Decision Tree", dt_auc),
    ("Random Forest", rf_auc),
    ("SVM", svm_auc)
]

for model, score in results:
    print(f"{model}: {score}")

Logistic Regression: 0.8529618890230132
Decision Tree: 0.9934791820944405
Random Forest: 0.9978693621799106
SVM: 0.8531343665660119


### Step 10: Final Comparison & Conclusion
In this final step, we compare **Training Accuracy** vs **Test Accuracy**.
*   If Training is much higher than Test, the model is **Overfitting** (memorizing).
*   If they are close and high, the model is **Robust** and ready for production.

In [32]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Initialize evaluator for accuracy
acc_eval = MulticlassClassificationEvaluator(labelCol="churn", metricName="accuracy")

# List of models and their training predictions
models = [
    ("Logistic Regression", lr_model.transform(train_df), lr_pred),
    ("Decision Tree", dt_model.transform(train_df), dt_pred),
    ("Random Forest", rf_model.transform(train_df), rf_pred),
    ("SVM", svm_model.transform(train_df), svm_pred)
]

print(f"{'Model':<20} | {'Train Accuracy':<15} | {'Test Accuracy':<15}")
print("-" * 55)

for name, train_p, test_p in models:
    train_acc = acc_eval.evaluate(train_p)
    test_acc = acc_eval.evaluate(test_p)
    print(f"{name:<20} | {train_acc:<15.4f} | {test_acc:<15.4f}")

Model                | Train Accuracy  | Test Accuracy  
-------------------------------------------------------
Logistic Regression  | 0.7657          | 0.7666         
Decision Tree        | 0.9906          | 0.9900         
Random Forest        | 0.9963          | 0.9963         
SVM                  | 0.7662          | 0.7664         
